# A tutorial on Huggingface Datasets class

We'll play with a NatureLM dataset to understand how this can help us

## First, install this repository
Go to the root and run
```uv sync```

also, run ```uv pip install -e .```

In [2]:
from datasets import load_dataset

## We're going to connect to a jsonl dataset and stream the rows!

In [3]:
data_path = "gs://foundation-model-data/data/compiled-datasets/v1.3/s2_eval_light_valid.jsonl"

#### We need this for connecting to gcp

In [4]:
storage_options = {"project": "okapi-274503"}

#### The 'json' keyword tells 🤗 load_dataset that the data is json/jsonl oriented
The streaming parameter must be set to True for streaming data

In [9]:
ds = load_dataset("json", data_files=data_path, storage_options=storage_options, streaming=True, split="train")

In [10]:
type(ds)

datasets.iterable_dataset.IterableDataset

#### The thing with an iterable dataset is that you can't index into it and you can't know its length!

In [11]:
len(ds)

TypeError: object of type 'IterableDataset' has no len()

#### But we can nicely iterate over the data

Let's do that for the first 10 samples / rows / records

In [14]:
%%time
records = []

for i, row in enumerate(ds):
    records.append(row)
    if i == 9:
        break

CPU times: user 404 ms, sys: 537 ms, total: 941 ms
Wall time: 5.84 s


In [15]:
len(records)

10

In [16]:
records[0]

{'prompt': "<Audio><AudioHere></Audio> Which of these, if any, are present in the audio recording? Vermilion Flycatcher, Tufted Titmouse, Southern Chestnut-tailed Antbird, Giant Ibis, Eurasian Magpie, Yucatan Vireo, Tropical Mockingbird, Southern Antpipit, Yellow-browed Shrike-Vireo, Great Parrotbill, Brown Tinamou, European Common Starling, Southern Yellowthroat, Marsh Tit, Taiwan Barbet, American Kestrel, Black-headed Parrot, Güldenstädt's Redstart, Rufous-tailed Scrub Robin, Helmeted Curassow, Northern Flicker, Varied Thrush, Madagascar Ibis, Garden Warbler, Carolina Wren, Common Grackle, Golden Parrotbill, Great Tit, European Goldfinch, Black-girdled Barbet, Eastern Meadowlark, Great Spotted Cuckoo, Rufous-sided Crake, Striated Pardalote, Common Chaffinch",
 'path': '/home/ubuntu/foundation-model-storage/foundation-model-data/audio_16k/noise/wham_noise/01ec021e_2.1985_026o0302_-2.1985.wav',
 'text': 'None',
 'task': 'species-sci-options-detection',
 'files': None}

### we can also shuffle the dataset before iterating
this is a *pseudo-shuffle* because it depends upon on a buffer_size. That's how many samples are randomly chosen from
to fill a buffer, then, after iterating through those the samples are replaced by the next buffer_size batch.

In [17]:
shuffled_dataset = ds.shuffle(seed=42, buffer_size=10_000)

In [19]:
%%time
records = []

for i, row in enumerate(shuffled_dataset):
    records.append(row)
    if i == 9:
        break

records[0]

CPU times: user 375 ms, sys: 389 ms, total: 764 ms
Wall time: 3.74 s


{'prompt': 'This is a list of reference recordings of the Barred Buttonquail. Recording A: <Audio><AudioHere></Audio> Recording B: <Audio><AudioHere></Audio> Recording C: <Audio><AudioHere></Audio> This a recording of an unknown species. <Audio><AudioHere></Audio> Is the query recording of the same species as the reference recordings? Use the common name for the species.',
 'path': None,
 'text': 'The query species is Pale-headed Rosella. No.',
 'task': 'classification_common',
 'files': ['/home/ubuntu/foundation-model-storage/foundation-model-data/audio_16k/animalspeak2/16khz/Xeno-canto/XC234422-554982_47-07sp-renw%20mp3.flac',
  '/home/ubuntu/foundation-model-storage/foundation-model-data/audio_16k/animalspeak2/16khz/iNaturalist/485037.flac',
  '/home/ubuntu/foundation-model-storage/foundation-model-data/audio_16k/animalspeak2/16khz/Xeno-canto/XC113376-BARRED%20BUTTON%20-GREY-BREASTED%20QUAIL%2014.flac',
  '/home/ubuntu/foundation-model-storage/foundation-model-data/audio_16k/animals

### We can also filter or apply functions to a streaming dataset!

In [20]:
def add_prefix(example):
    example["text"] = "My text: " + example["text"]
    return example

In [21]:
# this operation doesn't actually apply the function yet....
new_ds = ds.map(add_prefix)

In [22]:
%%time
records = []

for i, row in enumerate(new_ds):
    records.append(row)
    if i == 1:
        break

records[0]

CPU times: user 323 ms, sys: 417 ms, total: 740 ms
Wall time: 4.59 s


{'prompt': "<Audio><AudioHere></Audio> Which of these, if any, are present in the audio recording? Vermilion Flycatcher, Tufted Titmouse, Southern Chestnut-tailed Antbird, Giant Ibis, Eurasian Magpie, Yucatan Vireo, Tropical Mockingbird, Southern Antpipit, Yellow-browed Shrike-Vireo, Great Parrotbill, Brown Tinamou, European Common Starling, Southern Yellowthroat, Marsh Tit, Taiwan Barbet, American Kestrel, Black-headed Parrot, Güldenstädt's Redstart, Rufous-tailed Scrub Robin, Helmeted Curassow, Northern Flicker, Varied Thrush, Madagascar Ibis, Garden Warbler, Carolina Wren, Common Grackle, Golden Parrotbill, Great Tit, European Goldfinch, Black-girdled Barbet, Eastern Meadowlark, Great Spotted Cuckoo, Rufous-sided Crake, Striated Pardalote, Common Chaffinch",
 'path': '/home/ubuntu/foundation-model-storage/foundation-model-data/audio_16k/noise/wham_noise/01ec021e_2.1985_026o0302_-2.1985.wav',
 'text': 'My text: None',
 'task': 'species-sci-options-detection',
 'files': None}

# 🤗

# For more info on things like shuffling data
see https://huggingface.co/docs/datasets/en/stream 